Ch 17-18 Dynamic Programming, Hard, and Medium Problems --
[Open in Colab](https://colab.research.google.com/github/henry4j/-/blob/main/episode_17.ipynb)
* [Ch 1-10: Array, Strings, Lists, Stacks, Queues, Tree, Graph, Algebra, Search](https://colab.research.google.com/github/henry4j/-/blob/main/episode_1A.ipynb)

In [158]:
# @title import before you begin
from functools import lru_cache, reduce
from itertools import accumulate, islice, chain, count, pairwise, tee
from collections import namedtuple, Counter, defaultdict, deque
from collections.abc import Mapping
from dataclasses import dataclass
from enum import Enum, IntFlag
from heapq import heappush, heappop, heapify
from typing import Optional, Union
from math import *
from operator import itemgetter
from random import *
from sys import maxsize as inf  # infinity: 2**63-1.
import bisect
import more_itertools
import operator

class BTree:
  def __init__(self, key=None, left=None, right=None, value=None):
    self.key, self.left, self.right, self.value_ = key, left, right, value

  @property
  def value(self):
    return self.key if self.value_ is None else self.value_

def renumerate(it, stop=None):
  return (L := list(it)).reverse() or zip(count(stop or len(L), -1), L)

def recap(kv, k, v):
  return (kv[0] + k, kv[1] + v)

def dedupe(iterable):
  seen = set()
  for e in iterable:
    if e not in seen:
      seen.add(e)
      yield e

# def maxima(iterable, key=None): return minima(iterable, key, inverse=True)
def maxima(iterable, key=lambda e: e):
  maxima, max_key = None, None
  for k, e in zip(map(key, (it := tee(iterable, 2))[0]), it[1]):
    if not maxima or k > max_key:
      maxima, max_key = [e], k
    elif k == max_key:
      maxima.append(e)
  return maxima

def minima(iterable, key=lambda e: e, inverse=False):
  iterator = iter(iterable)
  minima = [next(iterator, None)]
  min_key = minima and key(minima[0])
  lt_ = operator.gt if inverse else operator.lt
  for e in iterator:
    if lt_((k := key(e)), min_key):
      minima, min_key = [e], k
    elif k == min_key:
      minima.append(e)
  return minima

Cut = namedtuple("Cut", "start stop", defaults=[0, 0])
Cut.off = classmethod(lambda cls, stop: cls(start=0, stop=stop))
Cut.__abs__ = lambda self: abs(self.stop - self.start)
Cut.__int__ = lambda self: self.stop - self.start
Cut.__call__ = lambda self, seq: seq[self.start:self.stop]

Intvl = namedtuple("Intvl", "l_end, r_end")  # left-, and right-closed ends.
Intvl.__abs__ = lambda self: abs(self.r_end+1 - self.l_end)

def __repr__(obj):
  unmap = lambda v: set(v.keys()) if isinstance(v, Mapping) else v
  m = len(values := list(unmap(v) for (k, v) in vars(obj).items() if k[0] != "_"))
  n = next((i for i, e in enumerate(reversed(values)) if e is not None), m)
  return f"{obj.__class__.__name__}({repr(values[:m-n])[1:-1].replace('{', '_={')})"

@dataclass
class SNode:
  value: int = None
  next: "SNode" = None
  #
  def __iter__(self):
    L = self
    while L:
      yield L
      L = L.next

  @classmethod
  def from_values(cls, *values):
    L = None
    for value in reversed(values):
      L = cls(value, L)
    return L  # the head node of the list.

  @classmethod
  def pool(cls, *values):
    push = lambda acc, new: setattr(new, "next", acc) or new
    # return deque(accumulate(map(cls, reversed(values)), push), maxlen=1).pop()
    return reduce(push, map(cls, reversed(values)))

def btree_from_values(cls, values, start=0, stop=None):
  if stop is None:
    stop = len(values)
  if stop - start>0:
    mid = (start+stop-1) // 2  # mid becomes start, when stop-start-2.
    L = cls.from_values(values, start, mid)
    R = cls.from_values(values, mid+1, stop)
    return cls(values[mid], L, R, None)

class BTree:
  def __init__(self, key=None, left=None, right=None, value=None):
    self.key, self.left, self.right, self.value_ = key, left, right, value

  @property
  def value(self):
    return self.key if self.value_ is None else self.value_

BTree.__repr__ = __repr__  # yapf:disable
BTree.from_values = classmethod(btree_from_values)

In [153]:
# @title ##### def three_sum_closest(L, query=0) -- i, j, k: [0,n-2), ≥i+1, ≤n-1
def three_sum(L, query=0):
  L, n = sorted(L), len(L)
  for i in range(n-2):
    j, k = i+1, n-1
    while j < k:
      diff = sum((L[i], L[j], L[k])) - query
      if diff<0:
        j += 1
      elif diff>0:
        k -= 1
      else:
        yield (L[i], L[j], L[k])
        j, k = j+1, k-1

def three_sum_closest(L, q=0):
  L, n = sorted(L), len(L)
  minima = None
  for i in range(n-2):
    j, k = i+1, n-1
    while j < k:
      diff = sum((L[i], L[j], L[k])) - q
      if not minima or abs(diff) < min_diff:
        minima, min_diff = [(L[i], L[j], L[k])], abs(diff)
      elif abs(diff) == min_diff:
        minima.add((L[i], L[j], L[k]))
      if diff<0:
        j += 1
      elif diff>0:
        k -= 1
      else:
        j, k = j+1, k-1
  return minima

assert {(-1, -1, 2), (-1, 0, 1)} == set(three_sum([-1, 0, 1, 2, -1, -4], 0))
assert [(-4, 1, 2)] == three_sum_closest([-1, 2, 1, -4], 0)

!! **fnmatch, rematch** Write a function that matches UNIX filenames.

In [ ]:
def match1(t, p, any_char):  # any_char: "?" (fnmatch), or "." (rematch)
  return p[0] == any_char or p[0] == t[0]

def fnmatch(t, p):  # P: pattern with "?", and "*".
  if p == "":  # base cases
    return t == ""
  # recursive cases:
  if p[0] == "*":
    return fnmatch(t, p[1:]) or t and fnmatch(t[1:], p)
  elif t and match1(t, p, "?"):
    return fnmatch(t[1:], p[1:])

def rematch(t, p):  # P: pattern with "." and "*".
  if p == "":  # base case:
    return t == ""
  # recursive cases:
  if len(p)>1 and p[1] == "*":
    return rematch(t, p[2:]) or t and match1(t, p, ".") and rematch(t[1:], p)
  elif t and match1(t, p, "."):
    return rematch(t[1:], p[1:])

xyz = "xyzz"
assert fnmatch(xyz, "*")
assert fnmatch(xyz, "**")
assert fnmatch("", "") and not fnmatch("x", "") and not fnmatch("", "x")
assert fnmatch(xyz, "*z") and fnmatch(xyz, "x*z") and fnmatch(xyz, "x*")
assert rematch("abc", ".b.") and rematch("abc", "a.c")
assert not rematch("aaa", "*") and rematch("aaa", "a*")
assert rematch("aaa", "aaa*") and rematch("abc", "a*.*c*")

!! **LP subsequence, LC substring, LC subsequence, Minimal # partitions, Longest increasing subsequence** Find the minimum # of partitions to make all partitions palindromic?

In [ ]:
def lpsubseq(L):  # LPS, longest palindromic subsequence http://youtu.be/Mbav2iuJ7xQ.
  def dp(start, stop):  # begin(b), end(e)
    if stop - start<3:
      if L[start] == L[stop-1]:
        yield L[start:stop]
      else:
        yield from (L[start:start+1], L[start+1:stop])
    elif L[start] == L[stop-1]:
      yield from (L[start] + sseq + L[stop-1] for sseq in dp(start+1, stop-1))
    else:
      yield from dedupe(
          maxima(chain(dp(start+1, stop), dp(start, stop-1)), key=len))

  return dp(0, len(L))

def lcsubseq(L, R):  # LCS, longest common subsequence, https://wikipedia.org/wiki/longest_common_subsequence yapf:disable
  def dp(m, n):
    if m==0 or n==0:
      yield ""
    elif L[m-1] == R[n-1]:
      yield from (subseq + L[m-1] for subseq in dp(m-1, n-1))
    else:
      yield from dedupe(maxima(chain(dp(m-1, n), dp(m, n-1)), key=len))

  return dp(len(L), len(R))

def lcsubstring(L, R):  # LCS, longest common substring, https://leetcode.com/problems/maximum-length-of-repeated-subarray/ yapf:disable
  def lcsuff(m, n):
    if m>0 and n>0 and L[m-1] == R[n-1]:
      start, stop = lcsuff(m-1, n-1)
      return Cut(start, stop+1) if stop > start else Cut(m-1, m)
    return Cut(0, 0)

  cases = (lcsuff(m, n) for m in range(len(L)+1) for n in range(len(R)+1))
  return tuple(cut(L) for cut in maxima(cases, key=abs))

def lisubsequence(L):  # LIS, longest increasing subsequence.
  def lisubseq(i):  # longest increasing subsequence ending at index i.
    cases = chain([(L[i],)], (
        subseq + (L[i],)  #
        for j in range(i)
        for subseq in lisubseq(j)
        if subseq[-1] <= L[i]))  # 0 <= j < i
    return maxima(cases, key=len)

  return maxima((subseq for i in range(len(L)) for subseq in lisubseq(i)), key=len)

def lisubseq_bottom_up(L, debug=False):
  memos = []
  for i, e in enumerate(L):
    for j, subseq in reversed(list(enumerate(memos))):
      if subseq[-1] <= L[i]:
        break
    else:
      j = -1
    subseq = memos[j] + [L[i]] if j != -1 else [L[i]]
    if j < len(memos)-1:
      memos[j+1] = subseq
    else:
      memos.append(subseq)
    _ = debug and print(f"{memos=}")
  return memos[-1]

assert ['abzba'] == list(lpsubseq('xaybzba'))
assert ['bab', 'aba'] == list(lpsubseq('abab'))
assert ("aba", "bab") == lcsubstring("abab", "baba")
assert ([3, 2, 1],) == lcsubstring([1, 2, 3, 2, 1], [3, 2, 1, 4, 7])
assert ("abace",) == tuple(lcsubseq("apbcadcqer", "rasbtaucve"))
assert [(9,), (8,), (7,)] == lisubsequence([9, 8, 7])
assert [(1, 5, 6), (1, 2, 3)] == lisubsequence((7, 8, 1, 5, 6, 2, 3))
assert [1, 2, 3] == lisubseq_bottom_up([7, 8, 1, 5, 6, 2, 3], debug=False)
# debug output:
# memos=[[7]]
# memos=[[7], [7, 8]]
# memos=[[1], [7, 8]]
# memos=[[1], [1, 5]]
# memos=[[1], [1, 5], [1, 5, 6]]
# memos=[[1], [1, 2], [1, 5, 6]]
# memos=[[1], [1, 2], [1, 2, 3]]

In [ ]:
# Find the optimal buying and selling strategy for k stock trades over n days.
def maximize_profit_of_stock_trades(L, k):  # at most k trades.
  def prog(d, k):  # d day stock prices, at most k trades.
    recap = lambda kv, k, v: (kv[0] + k, kv[1] + v)
    cases = (
        recap(prog(b, k-1), gain, ((b, s),))
        for b in range(0, d-1)
        for s in range(b+1, d)
        if k>0 and (gain := L[s] - L[b])>0)
    return max(cases, default=(0, ()))

  return prog(len(L), k)

L = [10, 5, 15, 20, 12, 18, 25, 10, 15]
assert (33, ((1, 3), (4, 6), (7, 8))) == maximize_profit_of_stock_trades(L, 3)
assert (0, ()) == maximize_profit_of_stock_trades([2, 1], 3)

In [ ]:
# Divide a shelf of books into at most k contiguous partitions, ensuring the maximum total pages in a partition is minimized.
def bookshelf(books, k):  # at most k partitions.
  book_acc = list(accumulate(books, initial=0))
  book_sum = lambda start, stop: book_acc[stop] - book_acc[start]
  def prog(m, k):  # m books to process, at most k partitions.
    if m>0 and k>0:
      cases = (prog(n, k-1) + ((n, m),) for n in range(m))
      return min(cases, key=lambda e: max(e, key=lambda e: book_sum(e[0], e[1])))
    else:
      return ()

  return prog(len(books), k)

books = [1, 2, 3, 4, 5, 6, 7, 8, 9]
assert ((0, 6), (6, 8), (8, 9)) == bookshelf(books, 3)

In [ ]:
def knapsack01(skus, capacity):  # a seq of costs & values.
  @lru_cache(maxsize=None)  # by capacity c and n items.
  def prog(n, c):  # returns a tuple of (a yield, items).
    if n==0:
      return (0, ())  # a tuple of ($0, none items).
    sku = skus[n-1]
    cases = (recap(prog(n-1, c - cost), value, seq)
             for (cost, value), seq in ((sku, (n-1,)), ((0, 0), ()))
             if c >= cost)  # at most two cases
    return max(cases, key=lambda kv: kv[0])

  return prog(len(skus), capacity)

def knapsack_unbounded(skus, capacity):
  @lru_cache(maxsize=None)
  def prog(c):
    cases = (recap(prog(c - sku[0]), sku[1], (i,))
             for i, sku in enumerate(skus)
             if c >= sku[0])
    return max(cases, key=lambda kv: kv[0], default=(0, ()))

  return prog(capacity)

skus = [(2, 2), (1, 1), (4, 10), (1, 2), (12, 4)]
assert (15, (0, 1, 2, 3)) == knapsack01(skus, 16)
assert (36, (3, 3, 3, 2, 2, 2)) == knapsack_unbounded(skus, 15)

16.1 Write a program to swap two numbers in-place without using additional variables.  
16.2 Write a program to find the frequency of occurrences of any given word in a book.  
16.3 Given two line segments, write a program to identify the point, where they cross each other.  
16.4 Write a program to test if a player has won a tic-tac-toe game.  
16.5 Write a method to compute the number of trailing zeros in n factorial.
```python
zeros = lambda n: (q := n // 5) + (zeros(q) if q > 4 else 0)  # quotient!
assert (0, 1, 4, 6, 18) == tuple(zeros(n) for n in (4, 5, 24, 25, 75))
```
16.6 Given two integer arrays, find the pair of integers (one from each array) with the smallest absolute difference. How about sorting and scanning two arrays?  
16.7 Write a program that finds the maximum of two numbers. Do not use if-else and comparisons.  

In [ ]:
# @title ##### 16.8 Spell out English words for numerical figures. Write "One Thousand Two Hundred Thirty Four" for 1234.
digits = [None, "One", "Two", "Three", "Four", "Five", "Six", "Seven", "Eight", "Nine"]
teens = [
  "Ten",
  "Eleven",
  "Twelve",
  "Thirteen",
  "Fourteen",
  "Fifteen",
  "Sixteen",
  "Seventeen",
  "Eighteen",
  "Nineteen",
]
tens = [
  None,
  "Ten",
  "Twenty",
  "Thirty",
  "Forty",
  "Fifty",
  "Sixty",
  "Seventy",
  "Eighty",
  "Ninety",
]
bigs = [None, "Thousand", "Million"]

def read(d, x1000=0):
  if d < 0:
    return ("Negative",) + read(-decimal, x1000)
  if d == 0:
    return ("Zero",) if x1000 == 0 else ()
  return (
    read(d // 1000, x1000 + 1)
    + read3d(d % 1000)
    + ((bigs[x1000],) if x1000 > 0 else ())
  )

def read3d(d):
  if d == 0:
    return ()
  if d >= 100:
    return (digits[d // 100], "Hundred") + read3d(d % 100)
  if 10 < d < 20:
    return (teens[d - 10],)
  if d == 10 or d > 19:
    return (tens[d // 10],) + read3d(d % 10)
  return (digits[d],)

# fmt:off
assert ('Zero',) == read(0)
assert ('Ten', 'Million', 'Two', 'Hundred', 'Twelve', 'Thousand', 'Twenty', 'One') == read(10212021)

16.9 Create functions to perform multiplication, subtraction, and division on integers. You can only use additions.

16.10 Given a list of people and their birth and death years, calculate the year with the maximum number of living individuals.



In [ ]:
# @title ##### 16.11 Given two different lengths of wooden planks, short and long, and a specific number of planks K, find all possible total lengths of a diving board that can be constructed using exactly K planks. Each plank used must be either short or long.
def diving_boards(k: int, short: int, long_: int):
  @lru_cache(maxsize=None)
  def prog(k_planks: int, s_planks, l_planks):
    if k_planks==0:
      yield 0
    else:
      yield from chain(
          (e + short for e in prog(k_planks-1, s_planks+1, l_planks)),
          (e + long_ for e in prog(k_planks-1, s_planks, l_planks+1)),
      )

  yield from prog(k, 0, 0)

assert [6, 7, 8, 9] == list(diving_boards(3, 2, 3))

16.12 Given an XML document and a mapping of XML tags to integer values, write a program that compresses an XML doc with encoding rules.
* Element: Tag Attributes END Children END
* Attribute: Tag Value
* END: 0
* Tag: A predefined integer mapping, e.g., family -> 1, person ->2, firstName -> 3, lastName -> 4, state
* Value: A string value


16.13 Given two squares on a 2D plane, find a line that would cut these two squares in half. Each square has top and bottom sides running parallel to the x-axis.  
16.14 Given a set of points on a 2D plane, find the line that passes through the maximum number of points.  

In [ ]:
# @title ##### 16.15 Imagine a secret code with four colors: Red (R), Yellow (Y), Green (G), and Blue (B). A game player tries to guess the secret code by picking four colors. Scores are given as a hit, if the color and the place both are right, or a pseudo-hit, if the color is right, but the place is not right.
def score(codes, guess):
  i_hits = {i for i, c, g in zip(range(4), codes, guess) if c == g}
  c_codes = Counter(c for i, c in enumerate(codes) if i not in i_hits)
  c_guess = Counter(g for i, g in enumerate(guess) if i not in i_hits)
  return len(i_hits), sum(min(h, c_codes.get(g, 0)) for g, h in c_guess.items())

score("RGBY", "GGRR")

!! **slice unsorted subarray, sum rainwater**

16.16 Given an array, find a subarray that, when sorted, sorts the entire array.  
17.21: Given n non-negative integers representing an elevation map where the width of each bar is 1, compute how much water it is able to trap after raining. For example, given [0,1,0,2,1,0,1,3,2,1,2,1], return 6.

In [ ]:
def slice_unsorted_subarray(*L):  # scan forward, backward to see where to start, stop.
  def renumerate(it, stop=None):  # stop: defaults to the length of iterable.
    (R := list(it)).reverse()
    return zip(count(stop or len(R), -1), R)

  L_cummax = zip(L, accumulate(L, max))  # to see max_a: max ahead.
  stop = next((stop for stop, (e, max_a) in renumerate(L_cummax) if e < max_a), 0)
  L_cummin = zip(L, reversed(list(accumulate(reversed(L), min))))  # min_a: min ahead.
  start = next((start for start, (e, min_a) in enumerate(L_cummin) if e > min_a), 0)
  return slice(start, stop)

def sum_rainwater(E):  # L: elevations, also called heights.
  cummax_behind = accumulate(E, max)  # O(1) space to look behind.
  cummax_back = list(accumulate(reversed(E), max))  # O(n) space to look ahead.
  cummax_ahead = reversed(cummax_back)
  return sum([
      min(*maxima_behind_ahead) - elev  # water in a column: min(maxima) -height.
      for (elev, *maxima_behind_ahead) in zip(E, cummax_behind, cummax_ahead)
  ])

assert 6 == sum_rainwater([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1])

assert slice(0, 0) == slice_unsorted_subarray()
assert slice(0, 0) == slice_unsorted_subarray(1)
assert slice(0, 0) == slice_unsorted_subarray(1, 3)
assert slice(0, 2) == slice_unsorted_subarray(2, 1, 3)
assert slice(1, 4) == slice_unsorted_subarray(0, 2, 3, 1, 4)
assert slice(3, 5) == slice_unsorted_subarray(0, 1, 1, 2, 1)

In [ ]:
# @title ##### 16.17 Given an array of integers (both positive and negative), write a program to find the max sum sub-array (contiguous sequence).

In [ ]:
# @title ##### 16.18 Pattern Matching : You are given two strings, pattern and value.The patternstring consists of just the letters aand b, describing a pattern within a string. For example, the string catcatgocatgomatches the pattern aabab(where catis aand gois b). It also matches patterns like a, ab, and b. Write a method to determine if value matches pattern.
def match(text: str, pattern: str):
  def prog(T, start, stop, **kwargs):
    if stop == start:
      if T == "":
        yield kwargs
    else:
      c = pattern[start]
      if lead := kwargs.get(c, None):
        if T.startswith(lead):
          yield from prog(T[len(lead):], start+1, stop, **kwargs)
      else:
        m = len(T)
        for k in range((m+1) // pattern[start:stop].count(c), 1, -1):
          lead, tail = T[:k], T[k:]
          yield from prog(tail, start+1, stop, **(kwargs | {c: lead}))

  yield from prog(text, 0, len(pattern))

# list(match("catcatgocatgo", "a"))
# list(match("catcatgocatgo", "ab"))
# list(match("catcatgocatgo", "aba"))
# list(match("catcatgocatgo", "aabab"))

16.19 Pond Sizes: You have an integer matrix representing a plot of land, where the value at that location
represents the height above sea level. A value of zero indicates water. A pond is a region of water
connected vertically, horizontally, or diagonally. The size of the pond is the total number of
connected water cells. Write a method to compute the sizes of all ponds in the matrix.
EXAMPLE
Input:
0 2 1 0
0 1 0 1
1 1 0 1
0 1 0 1
Output: 2, 4, 1 (in any order)

In [ ]:
# @title ##### 16.21 Given two integer arrays, find two integers (one from each array) that can be swapped to equalize their sums.
lhs, rhs = [1, 2, 3, 5], [4, 5, 6]
q, r = divmod(sum(lhs) - sum(rhs), 2)
assert r == 0
matcher, seen = {q+e for e in rhs}, set()
for e in lhs:
  if e in matcher and e not in seen:
    print((e, e-q))
    seen |= {e}

In [ ]:
# @title ##### 16.22 An ant starts on an infinite grid of alternating white and black squares, facing right. At each step:
# On a white square, it flips the color, turns 90° clockwise, and moves forward.
# On a black square, it flips the color, turns 90° counterclockwise, and moves forward.
# Write a program to simulate the ant's movement for K steps and print the final grid configuration.
# The grid structure must be designed by you. The input is the integer K, and the program should print the grid without returning anything.
# A possible method signature is void printKMoves(int K).

In [ ]:
# @title ##### 16.23 Given a function rand5() that generates a random number between 1 and 5 (inclusive), write a function that generates a random number between 1 and 7 (inclusive).
def rand5():
  import random
  return random.randrange(5) + 1

def rand7():
  rand25 = 22
  while rand25 > 21:  # [1, 21]
    rand25 = 5 * (rand5() - 1) + rand5()  # [1, 25]
  return rand25 % 7 + 1  # [1, 7].

from collections import Counter
c = Counter(rand7() for _ in range(7000))
assert all(900 < e < 1100 for e in c.values())

In [ ]:
# @title ##### 16.24 Write a program to find all pairs from an integer array, which sum to integer x.
def pairs(L, x):
  matcher = set()
  seen = set()
  for e in L:
    if e in matcher and e not in seen:
      pair = tuple(sorted((e, x - e)))
      seen |= {*pair}
      yield pair
    else:
      matcher |= {x - e}

assert [(1, 2), (0, 3)] == list(pairs([1, 2, 3, 1, 0, -1], 3))

!! **LRU cache** 16.25 Design an LRU cache with a max size, that evicts the least recently used when full. You can assume the keys are integers and the values are strings.

In [ ]:
class DNode:
  def __init__(self, value, pred=None, succ=None):
    self.value, self.pred, self.succ = value, pred, succ

class LRUCache:
  def __init__(self, capacity=1):
    self.capacity = capacity
    self.ha5h = {}
    self.head = self.tail = None
  def put(self, k, v):
    k in self.ha5h and self.pop_node(self.ha5h[k])
    self.append_node(DNode((k, v)))
    self.ha5h[k] = self.tail
    while len(self.ha5h) > self.capacity:
      del self.ha5h[self.pop_node(self.head).value[0]]
    return self
  def get(self, k):
    if k in self.ha5h:
      self.pop_node(self.ha5h[k])
      self.append_node(self.ha5h[k])
      return self.tail.value[1]
  def append_node(self, node):  # push at tail
    node.succ, node.pred = None, self.tail
    if self.tail:
      self.tail.succ = node
      self.tail = node
    else:
      self.head = self.tail = node
  def pop_node(self, node):
    if self.head != node:
      node.pred.succ = node.succ
    else:
      self.head = self.head.succ
      self.head.pred = None
    if self.tail != node:
      node.succ.pred = node.pred
    else:
      self.tail = self.tail.pred
      self.tail.succ = None
    return node

c = LRUCache(3).put(1, "a").put(2, "b").put(3, "c")
assert "a" == c.get(1) and "b" == c.get(2)
assert c.put(4, "d").get(3) is None
assert c.put(5, "e").get(1) is None
assert "b" == c.put(5, "e").get(2)

In [ ]:
# @title
class CircularBuffer:  # http://www.programiz.com/python-programming/exceptions
  def __init__(self, capacity):
    self._a = [None] * (capacity+1)
    self._head = self._tail = 0

  def enq(self, v):
    if self.full():
      raise RuntimeError("This buffer is full.")
    self._a[self._tail] = v
    self._tail = (self._tail+1) % len(self._a)
    return self

  def deq(self):
    if self.empty():
      raise RuntimeError("This buffer is empty.")
    v = self._a[self._head]
    self._head = (self._head+1) % len(self._a)
    return v

  def empty(self):
    return self._head == self._tail

  def full(self):
    return self._head == (self._tail+1) % len(self._a)

# yapf: disable
buffer = CircularBuffer(3)
assert buffer.empty() and not buffer.full()
assert buffer.enq(10).enq(20).enq(30).full()
try: buffer.enq(40)
except RuntimeError: pass
assert [10, 20, 30] == [buffer.deq() for _ in range(3)]
assert buffer.empty()
try:  buffer.deq()
except RuntimeError: pass

In [ ]:
# @title ##### 16.26 Calculate the result of a given arithmetic expression, which only includes positive integers and the basic operations of addition, subtraction, multiplication, and division.
ops = {"*": 1, "/": 1, "+": 2, "-": 2, "(": 3}

def postfix(a):
  Q, stack = [], []
  for e in a:
    if e == "(":
      stack.append(e)
    elif e == ")":
      while stack[-1] != "(":
        Q.append(stack.pop())
      stack.pop()
    elif e not in ops:
      Q.append(e)
    else:
      while stack and ops[stack[-1]] <= ops[e]:
        Q.append(stack.pop())
      stack.append(e)
  while stack:
    Q.append(stack.pop())
  return Q

def evaluate(Q):  # rpn Q: reverse polish notation.
  stack = []
  for e in Q:
    if e in ops:
      e = eval("{2}{1}{0}".format(stack.pop(), e, stack.pop()))
    stack.append(e)
  return stack.pop()

assert "234*+23+4*+" == "".join(postfix(list("2+3*4+(2+3)*4")))
assert 34 == evaluate(postfix(list("2+3*4+(2+3)*4")))

In [ ]:
# @title ##### 18.1 Write a functions that adds two numbers. You should not use the addition (+) arithmetic operator.
def sum_(a, b):  # a: ones, b: carries
  return sum_(a ^ b, (a & b) << 1) if b > 0 else a

assert 10 == sum_(7, 3)

!! **random shuffle, reservoir_samples, weighted choice**

18.2 Write a function to shuffle a deck of cards. Each of the 52! permutations of the deck has to be equally probable.  
18.3 Write a function to randomly sample m integers out of n integers. Each element must have equal probability of being chosen.

In [ ]:
def fisher_yates_shuffle(L):
  n = len(L)
  for i in range(n-1):
    j = i + random.randrange(n - i)  # j is in [i, n-1] from i + [0, n-i-1].
    L[j], L[i] = L[i], L[j]
  return L

def reservoir_samples(iterable, k):
  samples = []
  count = 0
  for e in iterable:
    count += 1
    if len(samples) < k:
      samples.append(e)
    elif (i := randrange(count)) < k:
      samples[i] = e
  return samples

def weighted_choice(weights):
  rand = randrange(sum(weights))
  return next(i for i, e in enumerate(accumulate(weights)) if e > rand)

assert 3 == len(reservoir_samples(iter(range(100)), 3))
weights = [20, 30, 50]
c = Counter(weighted_choice(weights) for _ in range(1000))
sorted(c.items(), key=lambda k: c[k])

In [ ]:
# @title ##### 17.5 Find the longest subarray that has an equal number of letters and digits.
def longest_subarray_with_alpha_nums_balanced(s):
  cumsum = accumulate(([-1, 1][c.isalpha()] for c in s), initial=0)
  maxima, max_key = [], 0
  indices = {}  # indices keyed by a cunsum, whenever you see the same cumsum.
  for stop, sum_ in enumerate(cumsum, 0):
    if (start := indices.get(sum_, -1)) > -1:
      if (key := stop - indices[sum_]) > max_key:
        maxima, max_key = [(start, stop)], key
      elif key == max_key:
        maxima.append((start, stop))
    else:
      indices[sum_] = stop
  return maxima

assert [(0, 4), (3, 7)] == longest_subarray_with_alpha_nums_balanced("a12b55c")

In [ ]:
# @title ##### 17.6 Write a function to count the number of 2s between O and n.
def count_2s_upto_e(b):  # 1 up to E+1, 20 up to E+2, 300 up to E+3.
  return b * 10 ** (b - 1)

def count_2s_upto(n):
  count, e, p = 0, 0, n  # preceding(p) precedes number(n).
  while p > 0:
    d = p % 10  # we look at a digit (rE+e) in each iteration.
    count += d * count_2s_upto_e(e)
    if d > 3:
      count += 10**e  # e.g., count += 100 for [200, 299], when p >299 and e=2
    if d == 2:
      count += n % 10**e + 1  # e.g., count += 56 for [200, 255], when n: x255, e: 2.
    p //= 10
    e += 1
  return count

assert 1 == count_2s_upto(9)
assert 20 == count_2s_upto(99)
assert 300 == count_2s_upto(999)
assert 3059 == count_2s_upto(6789)

In [ ]:
# @title ##### 17.7 Each year, the government releases a list of the 10,000 most common baby names and their frequencies. Some names, like "John" and "Jon," are variations of the same name but are listed separately. Given two lists—one with names and frequencies, and another with equivalent name pairs—design an algorithm to consolidate frequencies for equivalent names. If "John" is equivalent to "Jon," and "Jon" to "Johnny," all three are treated as equivalent (transitive and symmetric relationship). The final list should group equivalent names and use any as the primary name.
def consolidate_baby_names(counter, synonyms):
  id_map = {}
  for lhs, rhs in synonyms:
    lhs_id, rhs_id = id_map.setdefault(lhs, lhs), id_map.setdefault(rhs, rhs)
    if lhs_id != rhs_id:
      id_ = ":".join(sorted(lhs_id.split(":") + rhs_id.split(":")))
      for name in id_.split(":"):
        id_map[name] = id_
      lhs_count, rhs_count = counter.pop(lhs_id, 0), counter.pop(rhs_id, 0)
      counter[id_] = lhs_count + rhs_count
  return counter

counter = dict(john=15, jon=12, chris=13, kris=4, christopher=19)
synonyms = (("jon", "john"), ("chris", "kris"), ("chris", "christopher"))
consolidate_baby_names(counter, synonyms)

In [ ]:
# @title ##### 17.8 Write a program to design a circus of the largest tower of people standing atop one another's shoulders. For practical and aesthetic reasons, each person must be both shorter and lighter than the person below him or her

In [ ]:
# @title ##### 17.9 Write a program to find the k-th number such that the only prime factors are 3, 5, and 7.
def multiple_of(prime_factors):
  queues = {p: deque([p]) for p in prime_factors}
  while True:
    yield (x := min(q[0] for q in queues.values()))
    for prime, q in queues.items():
      if q[0] == x:
        q.popleft()
      q.append(prime * x)

assert [3, 5, 7] == list(islice(multiple_of((3, 5, 7)), 3))
assert [15, 21, 25] == list(islice(multiple_of((3, 5, 7)), 4, 7))

!! **boyer_moore_majority** 17.10 Given an array of positive integers, determine the majority, if any. A majority is an element that occurs more than half the times in the array.

In [ ]:
def boyer_moore_majority(L):
  candidate, votes = None, 0
  for e in L:
    if votes==0:
      candidate, votes = e, 1
    else:
      votes += 1 if e == candidate else -1
  return candidate if L.count(candidate) > len(L) // 2 else None

assert 1 == boyer_moore_majority([1, 1, 1, 1, 2, 3, 4])

!! **shortest text snippet, shortest superset subarray, longest unique subarray**

17.11 Find the shortest text snippet that contains K words out of a large text.  
17.18 Find the shortest superset subarray that contains another shorter array.

In [178]:
def shortest_text_snippet(k_indices):  # O(L *logK), L: length of k indices.
  # positions, e.g., [[0, 89, 130], [95, 123, 177, 199], [70, 105, 117]]
  k_indices = [deque(e) for e in k_indices]
  min_window = window = [e.popleft() for e in k_indices]  # [0, 95, 70]
  heapify(heap := [(index, k) for k, index in enumerate(window)])
  while k_indices[heap[0][1]]:  # k_indices[2] becomes empty after 117.
    _, k = heappop(heap)
    window[k] = k_indices[k].popleft()
    min_window = min(min_window, window, key=lambda w: max(w) - min(w))
    heappush(heap, (window[k], k))
  return min_window

# * https://leetcode.com/problems/minimum-window-substring/
def slice_shortest_superset_subarray(L, subset):  # *smallest superset window*
  def prog():
    from collections import Counter
    demand = Counter(subset)
    supply = Counter({k: 0 for k in demand})
    start = 0
    for stop, i in enumerate(L, 1):  # i: in, o: out of a window.
      if i in supply:
        supply[i] += i in supply
        if supply >= demand:
          while supply.get(o := L[start], 1) > demand[o]:  # over-supplied.
            supply[o] -= o in supply
            start += 1
          yield slice(start, stop)

  return tuple(L[e] for e in minima(prog(), key=lambda e: e.stop - e.start))

# https://leetcode.com/problems/longest-substring-without-repeating-characters/
def longest_unique_subarray(L):  # also called the longest distinct slice.
  def prog():
    last_indices = {}
    start = 0
    for stop, e in enumerate(L):  # e: element being processed.
      if e in last_indices:
        yield Cut(start, stop)
        while e in last_indices:
          del last_indices[L[start]]
          start += 1
      last_indices[e] = stop
    yield Cut(start, len(L))

  return tuple(cut(L) for cut in maxima(prog(), abs))

def longest_unique_subarray(L):  # also called the longest distinct slice.
  def prog():
    seen = set()
    start = 0
    for stop, e in enumerate(L):  # e: element being processed.
      if e in seen:
        yield slice(start, stop)
        while e in seen:
          seen.remove(L[start])
          start += 1
      seen.add(e)
    yield slice(start, len(L))

  return tuple(L[s] for s in maxima(prog(), lambda e: e.stop - e.start))

k_word_indices = [[0, 89, 130], [95, 123, 177, 199], [70, 105, 117]]
assert [130, 123, 117] == shortest_text_snippet(k_word_indices)

assert ("ab.a", "a.ba") == slice_shortest_superset_subarray("ab.a_a.ba", "aba")
L = [7, 5, 9, 0, 2, 1, 3, 5, 7, 9, 1, 1, 5, 8, 8, 9, 7]
assert ([5, 7, 9, 1], [9, 1, 1, 5]) == slice_shortest_superset_subarray(L, [5, 1, 9])

assert ("",) == longest_unique_subarray("")
assert ("a",) == longest_unique_subarray("a")
assert ("ab",) == longest_unique_subarray("ab")
assert ("abc", "cde") == longest_unique_subarray("aabccdee")

In [ ]:
# @title ##### 17.12 Write a program to convert a binary search tree to a doubly linked list. Keep the values in order while converting in-place.
def iterate(tree):  # returns an iterator.
  stack = []
  while tree or stack:
    if tree:
      stack.append(tree)
      tree = tree.left
    else:
      yield (tree := stack.pop())
      tree = tree.right

b10 = BTree.from_values(range(10))
it = iterate(b10)
head = pred = next(it)
for tree in it:
  tree.left, pred.right = pred, tree
  pred = tree

tree = head
while tree:
  print(tree.value, end=" ")
  tree = tree.right

In [ ]:
# @title ##### 17.13 Given a text with all spaces, punctuation, and capitalization removed, e.g., "iresetthecomputeri tstilldidntboot" from "I reset the computer. It still didn’t boot!". Respace the text while keeping most letters recognizable. Don't worry about capitalization or punctuation for now.
def re_space(text, vocas):
  from functools import lru_cache
  vocas = dict.fromkeys(sorted(vocas, key=len, reverse=True))
  recap = lambda kv, k, v: (kv[0] + k, kv[1] + v)  # recap of (minutes, sequence).

  @lru_cache(maxsize=None)
  def prog(m):
    if m>0:
      cases = [
          recap(prog(n), 0 if text[n:m] in vocas else m - n, (m,))
          for n in range(m-1, -1, -1)
      ]
      return min(cases, key=lambda e: e[0])
    else:
      return (0, ())

  compose = lambda c: (text[i>0 and c[i-1]:stop] for i, stop in enumerate(c))
  rubbish, stops = prog(len(text))
  return rubbish, " ".join(compose(stops))

vocas = ["you", "reset", "the", "computer", "still", "didnt", "boot"]
text = "youresetthecomputeritstilldidntboot"
assert (2, "you reset the computer i t still didnt boot") == re_space(text, vocas)

In [ ]:
# @title ##### 17.14 Write a program to find the smallest million numbers in billion numbers. Your computer can hold all one billion numbers.
def quicksort_k(a, k=0, start=0, stop=None, key=None):
  quickfind_k(a, k, start, stop, True, key)
  return a

# http://en.wikipedia.org/wiki/Selection_algorithm#Optimised_sorting_algorithms
def quickfind_k(a, k=0, start=0, stop=None, sort=False, key=None):
  if stop is None:
    stop = len(a)
  if stop - start > 1:
    pivot = partition(a, start, stop, key)
    if k < pivot or sort:
      quickfind_k(a, k, start, pivot, sort, key)
    if pivot < k:
      quickfind_k(a, k, pivot + 1, stop, sort, key)
  return a

def partition(a, start, stop=None, key=None):
  if stop is None:
    stop = len(a)
  if key is None:
    key = lambda e: e
  pivot = start + random.randrange(stop - start)
  a[pivot], a[start], pivot = a[start], a[pivot], start
  for i in range(start + 1, stop):
    if key(a[i]) < key(a[start]):
      pivot += 1
      a[pivot], a[i] = a[i], a[pivot]
  a[pivot], a[start] = a[start], a[pivot]
  return pivot

for _ in range(1000):
  assert [0, 1] == quickfind_k([9, 8, 7, 6, 5, 4, 3, 2, 1, 0], 1)[:2]
  assert 2 == quickfind_k([9, 8, 7, 6, 5, 4, 3, 2, 1, 0], 2)[2]
  assert 3 == quickfind_k([9, 8, 7, 6, 5, 4, 3, 2, 1, 0], 3)[3]
  assert [0, 1, 2, 3] == quicksort_k([9, 8, 7, 6, 5, 4, 3, 2, 1, 0], 3)[:4]

In [ ]:
# @title ##### 17.15 Longest Word: Given a list of words, write a program to find the longest word made of other words in the list.
def longest_composition(L):
  vocas = dict.fromkeys(sorted(L, key=len, reverse=True))
  @lru_cache(maxsize=None)
  def prog(word, m):
    if m > 0:
      n = next((n for n in range(m - 1, 0, -1) if word[n:m] in vocas), None)
      if n:
        if word[:n] in vocas:
          yield (n, m)
        else:
          yield from (e + (m,) for e in prog(word, n))
    else:
      yield ()
  compose = lambda c: [word[i > 0 and c[i - 1] : stop] for i, stop in enumerate(c)]
  for word in vocas:
    yield from (compose(stops) for stops in prog(word, len(word)))

L =  ["cat", "cats", "catdogcats", "catcats", "dog", "dogcatsdog", "ratcatdogcats"]  # fmt: skip
assert ["cat", "dog", "cats"] == next(longest_composition(L), None)

In [ ]:
# @title ##### 17.16 A popular masseuse receives a sequence of back-to-back appointment requests and is debating which ones to accept. She needs a 15-minute break between appointments and therefore she cannot accept any adjacent requests. Given a sequence of back-to-back appointment requests (all multiples of 15 minutes, none overlap, and none can be moved), find the optimal (highest total booked minutes) set the masseuse can honor. Return the number of minutes.
def masseuse(L):  # L: massage appointmts in minutes.
  recap = lambda kv, k, v: (kv[0] + k, kv[1] + v)  # recap of (minutes, sequence).
  def prog(n):  # n appts. to process.
    if n > 0:
      key, value = L[n - 1], (n - 1,)  # key: minutes, value: seq. of decisions.
      cases = (recap(prog(n - 2), key, value), recap(prog(n - 1), 0, ()))
      return max(cases, key=lambda e: e[0])
    else:  # no appts. to process.
      return 0, ()
  return prog(len(L))

L = [30, 15, 60, 75, 45, 15, 15, 45]  # a list of appointmts.
minutes, indices = masseuse(L)
assert (180, (30, 60, 45, 45)) == (minutes, tuple(L[i] for i in indices))

!! **rabin karp's string match in O(m+n)** Find all indices of a pattern in a long text, "ana" in "banana anaconda eats".

In [ ]:
# rabin karp's string match in O(m+n), not naive O(m x n) -- pattern(m), text(n).
def rabin_karp(pattern, text):
  (pttn, m), (text, n) = ((tuple(map(ord, e)), len(e)) for e in (pattern, text))
  hash_p = hash(pttn, 0, m)
  hash_t = None  # e.g., hash(text, 0, m)
  for i_text in range(n - m+1):
    hash_t = hash(text, i_text, i_text + m, hash_t)
    if hash_p == hash_t and pttn == text[i_text:i_text + m]:
      yield i_text

def hash(text, start, stop, hash=None):  # O(m), or O(1).
  if hash is None:
    return reduce(lambda h, c: h*31 + c, text[start:stop], 0)  # O(m).

  hash = (hash<<5) - hash
  new, old = text[stop-1], text[start-1]  # new and old character code points.
  return hash + new - old * 31**(stop - start)  # O(1).

assert (1, 3, 10) == tuple(rabin_karp("ana", "banana an anaconda eats."))

In [163]:
# @title ##### 17.17 Given a text T and a set of queries Q, write a program to search T for each query in Q.
# Related: the longest common substrings to a set of queries Q.
class Trie:
  def __init__(self, value=None, **_):
    self.value, self.children = value, defaultdict(lambda: Trie())
  def __setitem__(self, key, child):
    if len(key)>1:
      self.children[key[0]][key[1:]] = child
    else:
      self.children[key[0]] = child if isinstance(child, Trie) else Trie(child)
  def __getitem__(self, key):
    return (c := self.children.get(key[0], None)) and c[key[1:]] if key else self
  def values(self):
    if self.value is not None:
      yield self.value
    for child in self.children.values():
      yield from child.values()

Trie.__repr__ = __repr__

s, suffs = "bananas", Trie()  # a suffix tree is a trie of all the suffixes.
for i, _ in enumerate(s):
  suffs[s[i:]] = i
#
L = "ba na nas s bas".split(" ")
expected = [[0], [2, 4], [4], [6], None]
assert expected == [(t := suffs[q]) and list(t.values()) for q in L]
# Trie for autocomplete
trie = Trie()
vocabs = ["brace", "brazil", "bread", "brew", "brag"]
for i, e in enumerate(vocabs):
  trie[e] = i  # or Trie(i).
#
assert (0, 1, 4) == tuple(trie["bra"].values())  # see the words with prefix `bra`
assert (2, 3) == tuple(trie["bre"].values())  # see the words with prefix `bre`.

17.19 Given an integer array from 1 to N appearing exactly once,
except for one number that is missing. How can you find the missing number in O(N) time and 0(1) space? What if there were two numbers missing?

17.20 Write a program that can quickly answer a median value, while random numbers are being generated and offered (a median bag).

In [ ]:
# @title ##### 17.22 Given two words of equal length that are in a dictionary, write a method to transform one word into another word by changing only one letter at a time. The new word you get in each step must be in the dictionary.
def transform(source, sink, dictionary):
  import string
  def backtrack(candidate):
    if reduce_off(candidate):
      yield tuple(candidate)
    else:
      for e in expand_out(candidate):
        candidate.append(e)
        yield from backtrack(candidate)
        candidate.pop()
  def reduce_off(candidate):
    return candidate[-1] == sink
  def expand_out(candidate):
    entered = set(candidate)
    for i, e in enumerate(word0 := list(candidate[-1])):
      for c in set(string.ascii_uppercase):
        word0[i] = c
        y = "".join(word0)
        if y not in entered and y in dictionary:
          yield y
      word0[i] = e

  yield from backtrack([source])

dictionary = set("DAMP LAMP RAMP LIMP LIMO LITE LIME LIKE".split(" "))
# assert tuple("DAMP RAMP LAMP LIMP LIME LIKE".split(" ")) ==
next(transform("DAMP", "LIKE", dictionary))t

!! **maxsum submatrix, maxsum subarray, largest subsquare**

17.24 Given a square matrix of integers, write a program to find a sub-matrix with the largest sum.

17.23 Given a square matrix of black and white cells, write a program to find the maximum sub-square such that all four borders are filled with black pixels.



In [ ]:
def slice_maxsum_submatrix(g):
  cumsum_h = (accumulate(c, initial=0) for c in zip(*g))
  cumsum = tuple(tuple(accumulate(r, initial=0)) for r in zip(*cumsum_h))
  sum_2d = lambda start_r, start_c, stop_r, stop_c: (
      cumsum[stop_r][stop_c] - cumsum[stop_r][start_c] - cumsum[start_r][stop_c] +
      cumsum[start_r][start_c])

  max_sum, max_cut = 0, (0, 0, 0, 0)
  m, n = len(g), len(g[0])  # this loop runs in O(m x n) time given an m x n graph.
  for stop_r in range(1, m+1):
    for start_r in range(stop_r):
      for stop_c in range(1, n+1):
        for start_c in range(stop_c):
          if (sum_ := sum_2d(start_r, start_c, stop_r, stop_c)) > max_sum:
            max_sum, max_cut = sum_, (start_r, start_c, stop_r, stop_c)

  return (max_sum, max_cut)

# Kadane's algorithm http://en.wikipedia.org/wiki/maximum_subarray_problem
def slice_maxsum_subarray(L):  # returns (maxsum, (start, stop)).
  max_sum = sum_ = 0  # max vs. running sums.
  max_cut = (0, 0)
  for stop, e in enumerate(L, 1):
    if sum_>0:
      sum_ = sum_ + e  # keep the sum running.
    else:  # start over the sum running .
      sum_, start = e, stop-1
    if sum_ > max_sum:
      max_sum, max_cut = sum_, (start, stop)
  return (max_sum, max_cut)

g = [  # 4 x 3 matrix
    [ 1,  0, 1],
    [ 0, -1, 0],
    [ 1,  0, 1],
    [-5,  2, 5]
]  # yapf:disable
assert (5, (1, 5)) == slice_maxsum_subarray([-2, 1, 3, -3, 4, -2, -1, 3])
assert (9, (0, 1)) == slice_maxsum_subarray([9, -3, 3])
assert (9, (2, 3)) == slice_maxsum_subarray([3, -3, 9])
assert (0, (0, 0)) == slice_maxsum_subarray([-9])
assert (0, (0, 0)) == slice_maxsum_subarray([0])
assert (8, (0, 1, 4, 3)) == (8, (0, 1, 4, 3)) == slice_maxsum_submatrix(g)
assert (3, (0, 0, 1, 2)) == slice_maxsum_submatrix([
    [ 1,  2, -1],
    [-3, -1, -4],
    [ 1, -5,  2]
])  # yapf:disable

17.25 Given millions of words, write a program to create the largest possible rectangle of letters such that every row forms a word (reading left to right) and every column forms a word (reading top to bottom).



In [ ]:
# @title
def groupby(iterable, key=None, unique=False):
  d = {}
  if key is None:
    key = lambda e: e
  for e in iterable:
    k = key(e)
    if k in d:
      if unique:
        d[k].add(e)
      else:
        d[k].append(e)
    else:
      d[k] = set((e,)) if unique else [e]
  return d

def permutate(a, k=None, i=0):  # places values at indices from i to k-1.
  if k is None:
    k = len(a)
  if i == k:
    yield a[:k]
  else:
    for j in {a[j]: j for j in range(i, len(a))}.values():
      a[i], a[j] = a[j], a[i]
      yield from permutate(a, k, i + 1)
      a[i], a[j] = a[j], a[i]

assert {0: [2, 2], 1: [1, 3, 3, 3]} == groupby([1, 2, 2, 3, 3, 3], lambda e: e % 2)
assert {3: {"abc", "def"}, 4: {"wxyz"}} == groupby(["abc", "def", "wxyz"], len, True)
assert [[1, 1], [1, 2], [2, 1]] == sorted(permutate([1, 1, 2], 2))

def rectangle(words):
  d = groupby(words, len, unique=True)
  n = max(d.keys())
  area = n**2
  for a in range(area, 0, -1):
    for w in (w for w in range(n, 0, -1) if 0 == a % w):
      h = a // w  # height is area divided by width
      if w >= h and d.get(w) and len(d[w]) >= h:
        for p in permutate(list(d[w]), h):
          if all(
            ("".join((p[r][c] for r in range(h))) in d[h] for c in range(w))
          ):
            return p

words = set("a cat ate tea strafer taeniae resters antiflu fiefdom earlobe resumes schools coconut acacias oracle largest fingernails".split(" "))  # fmt: skip
assert 7 == len(rectangle(words))

17.26 Sparse Similarity: The similarity of two documents (each with distinct words) is defined to be the size of the intersection divided by the size of the union. For example, if the documents consist of integers, the similarity of {1, 5, 3} and {1, 7, 2, 3} is e. 4, because the intersection has size 2 and the union has size 5. We have a long list of documents (with distinct values and each with an associated ID) where the similarity is believed to be "sparse:'That is, any two arbitrarily selected documents are very likely to have similarity O. Design an algorithm that returns a list of pairs of document IDs and the associated similarity. Print only the pairs with similarity greater than O. Empty documents should not be printed at all. For simplicity, you may assume each document is represented as an array of distinct integers.